<a href="https://colab.research.google.com/github/Joy-Kimani/slm-arm64-benchmark/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation Pipeline for a 70M-Parameter English SLM

In this notebook, we build the data pipeline that turns raw text stories into binary token streams ready for pretraining a 70M-parameter Small Language Model (SLM).



## Step 1: Mount Google Drive and Configure the Output Directory

Before downloading data or training a tokenizer, we mount Google Drive into and create a dedicated project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
out_dir = '/content/drive/MyDrive/slm70m'
os.makedirs(out_dir, exist_ok=True)

Mounted at /content/drive


## Step 2: Import the Dataset

TinyStories is a synthetic dataset of short stories written with words a 3 to 4 year old would understand. We can get it from HuggingFace.

### Why TinyStories works well
Fast iteration and Clean train/validation split.

In [ ]:
!pip install datasets tokenizers

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")
print(ds)

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


## Step 3: Train a 16k Byte-Level BPE Tokenizer
This step details training a 16k Byte-Level BPE tokenizer on 500k stories with a <|endoftext|> special token to guarantee full UTF-8 byte coverage without <unk> fallbacks while optimizing the embedding parameter budget for a 70M SLM.

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

vocab_size = 16000
tok_train_stories = 500_000

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<|endoftext|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

tok_data = ds['train'].select(range(tok_train_stories))

def text_batches(split, batch_size=1000):
    for i in range(0, len(split), batch_size):
        yield split[i:i+batch_size]['text']

tokenizer.train_from_iterator(text_batches(tok_data), trainer=trainer, length=len(tok_data))
tokenizer.save(os.path.join(out_dir, 'tokenizer.json'))
print(tokenizer.get_vocab_size())

16000


In [ ]:
text = "Once upon a time there was a pumpkin."
ids = tokenizer.encode(text).ids
print(ids)
print(tokenizer.decode(ids))

[432, 448, 259, 396, 402, 282, 259, 3275, 14]
Once upon a time there was a pumpkin.


## Step 4: Tokenize the Dataset

Each story ends with an <|endoftext|> token so the model learns where a story stops. We write the token IDs straight to disk in chunks, and we also count the bytes of text, which we need later to turn the loss into bits per byte.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

eot = tokenizer.token_to_id("<|endoftext|>")

def write_bin(split, filename, batch_size=5000):
    n_tokens = 0
    n_bytes = 0
    with open(os.path.join(out_dir, filename), 'wb') as f:
        for i in tqdm(range(0, len(split), batch_size), desc=filename):
            texts = split[i:i+batch_size]['text']
            ids = []
            for e in tokenizer.encode_batch(texts):
                ids.extend(e.ids)
                ids.append(eot)
            np.array(ids, dtype=np.uint16).tofile(f)    # uint16 works because vocab_size < 65536
            n_tokens += len(ids)
            n_bytes += sum(len(t.encode('utf-8')) for t in texts)
    return n_tokens, n_bytes

train_tokens, train_bytes = write_bin(ds['train'], 'train.bin')
val_tokens, val_bytes = write_bin(ds['validation'], 'validation.bin')
print(train_tokens, val_tokens)

train.bin:   0%|          | 0/424 [00:00<?, ?it/s]

validation.bin:   0%|          | 0/5 [00:00<?, ?it/s]

463152428 4655638


## Step 5: Save the dataset info and check the files

In [ ]:
import json

meta = {
    'vocab_size': tokenizer.get_vocab_size(),
    'train_tokens': train_tokens,
    'val_tokens': val_tokens,
    'val_bytes_per_token': val_bytes / val_tokens,
}
json.dump(meta, open(os.path.join(out_dir, 'meta.json'), 'w'))
print(meta)

{'vocab_size': 16000, 'train_tokens': 463152428, 'val_tokens': 4655638, 'val_bytes_per_token': 4.127973437797355}


In [ ]:
data = np.memmap(os.path.join(out_dir, 'train.bin'), dtype=np.uint16, mode='r')
print(len(data))
print(tokenizer.decode(data[:100].tolist()))

463152428
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

